In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

In [7]:
model_df = pd.read_csv('nhl_model_data.csv')

In [9]:
model_df['home_team_win'] = model_df['home_goals_for'] > model_df['away_goals_for']
model_df['home_team_win'] = model_df['home_team_win'].astype(int)

In [12]:
feature_cols = [
    col for col in model_df.columns 
    if (
        col.startswith("home_rolling_") or col.startswith("away_rolling_")
    )
    and (
        col.endswith("_avg5") or col.endswith("_avg10")
    )   
]      

In [14]:
print(len(feature_cols))

48


In [15]:
model_df = model_df.dropna(subset=feature_cols).reset_index(drop=True)

X = model_df[feature_cols]
y = model_df['home_team_win']

print(X.shape)
print(y.shape)

(1292, 48)
(1292,)


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

log_reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)

preds = log_reg_pipeline.predict(X_test)
probs = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, preds))
print("ROC AUC:", roc_auc_score(y_test, probs))
print("Classification Report:\n", classification_report(y_test, preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

Accuracy: 0.4864864864864865
ROC AUC: 0.4777617502103618
Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.54      0.53       141
           1       0.43      0.42      0.43       118

    accuracy                           0.49       259
   macro avg       0.48      0.48      0.48       259
weighted avg       0.49      0.49      0.49       259

Confusion Matrix:
 [[76 65]
 [68 50]]


In [23]:
import numpy as np
from sklearn.metrics import accuracy_score

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": log_reg_pipeline.named_steps["model"].coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df.sort_values("abs_coefficient", ascending=False).head(25)

,feature,coefficient,abs_coefficient
9,home_rolling_giveaways_avg5,-0.247868,0.247868
21,home_rolling_giveaways_avg10,0.213348,0.213348
20,home_rolling_blocked_shots_avg10,-0.210394,0.210394
47,away_rolling_pims_avg10,-0.202979,0.202979
11,home_rolling_pims_avg5,-0.186405,0.186405
23,home_rolling_pims_avg10,0.179945,0.179945
12,home_rolling_goals_for_avg10,0.177486,0.177486
43,away_rolling_hits_avg10,0.173380,0.173380
4,home_rolling_faceoff_win_pct_avg5,-0.168748,0.168748
32,away_rolling_blocked_shots_avg5,-0.162288,0.162288
